# Manhattan Distance

Manhattan distance is named after Manhattan since it has grid-like roads. One interesting use case of this distance would be to estimate the travel distance for taxirides, in order to, for example, estimate the fees that one may have to charge the passenger.

In the `data` folder, there's a database with data from July 2025 the taxi rides in NYC. The original database, after converting to `csv`,  had 350,000 lines, so it would be too big to upload. However, even the current version (10% the size) would be interesting to work with.

After deciphering what the coordinates Pickup/Dropoff zone codes represent, one could easily start measuring the Manhattan distance between them, and compare with the `distance` column given. This would be helpful in figuring out the most accurate way of adjusting the estimates to fit real-world outcomes, and would serve as as a source for a 'price range'. 


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/taxi.csv")

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Keep only rows with required columns not null
needed = ["PULocationID","DOLocationID","trip_distance"]
df_clean = df.dropna(subset=["PULocationID","DOLocationID"]).copy()

# Coerce IDs (strip spaces that appear in sample)
df_clean["PULocationID"] = df_clean["PULocationID"].astype(str).str.strip().astype(int)
df_clean["DOLocationID"] = df_clean["DOLocationID"].astype(str).str.strip().astype(int)

df_clean = df_clean[df_clean["trip_distance"] > 0]
df_clean.head()


In [ ]:
# (Since we only have zone codes, we fabricate stable pseudo (x,y) coords.)
zone_ids = pd.Index(
    pd.unique(pd.concat([df_clean.PULocationID, df_clean.DOLocationID], ignore_index=True))
).sort_values()

rng = np.random.default_rng(seed=42)
# Spread zones on a rough grid: assign random integers in a fixed range
coords_array = rng.integers(low=0, high=50, size=(len(zone_ids), 2))
zone_to_xy = {z: coords_array[i] for i, z in enumerate(zone_ids)}

def zone_coord(z):
    return zone_to_xy[int(z)]

# Preview a few mappings
list(zone_to_xy.items())[:5]


In [ ]:
from distances.manh import manhattan_distance  # assumes function(vec1, vec2)

def manhattan_zone_distance(pu_zone, do_zone):
    return manhattan_distance(zone_coord(pu_zone), zone_coord(do_zone))

df_clean["manhattan_dummy"] = df_clean[["PULocationID","DOLocationID"]].apply(
    lambda r: manhattan_zone_distance(r.PULocationID, r.DOLocationID), axis=1
)

df_clean[["PULocationID","DOLocationID","trip_distance","manhattan_dummy"]].head()


In [ ]:
# Fit linear model trip_distance ≈ a * manhattan_dummy
mask = df_clean["manhattan_dummy"] > 0
subset = df_clean[mask]
a = (subset["trip_distance"] / subset["manhattan_dummy"]).median()

df_clean["manhattan_scaled"] = df_clean["manhattan_dummy"] * a

corr_raw = subset["trip_distance"].corr(subset["manhattan_dummy"])
corr_scaled = subset["trip_distance"].corr(df_clean.loc[mask,"manhattan_scaled"])

print(f"Median scale factor a = {a:.3f}")
print(f"Correlation raw dummy vs trip_distance:   {corr_raw:.3f}")
print(f"Correlation scaled dummy vs trip_distance: {corr_scaled:.3f}")

df_clean[["trip_distance","manhattan_dummy","manhattan_scaled"]].head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample = df_clean.sample(min(2000, len(df_clean)), random_state=0)
fig, ax = plt.subplots(1,2, figsize=(12,5))

sns.scatterplot(data=sample, x="manhattan_dummy", y="trip_distance", s=15, ax=ax[0])
ax[0].set_title("Raw dummy Manhattan vs trip_distance")
sns.scatterplot(data=sample, x="manhattan_scaled", y="trip_distance", s=15, ax=ax[1])
ax[1].set_title("Scaled dummy Manhattan vs trip_distance")
for a_ in ax:
    a_.set_xlabel("Estimated (miles)")
    a_.set_ylabel("Actual trip_distance")

plt.tight_layout()
plt.show()
# ...existing code...
# New cell: Build distance matrix across a sample of zones
from distances.distance_matrix import make_distance_matrix

sample_zones = zone_ids[:25]  # first 25 zones
zone_points = np.vstack([zone_to_xy[z] for z in sample_zones])

dm_manhattan = make_distance_matrix(zone_points, metric="manhattan")
dm_manhattan[:5,:5]

In [ ]:
import seaborn as sns
import pandas as pd

dm_df = pd.DataFrame(dm_manhattan, index=sample_zones, columns=sample_zones)
plt.figure(figsize=(7,6))
sns.heatmap(dm_df, cmap="magma", cbar_kws={"label":"Manhattan distance (grid units)"})
plt.title("Dummy Manhattan Distance Between Sample Taxi Zones")
plt.xlabel("Zone ID")
plt.ylabel("Zone ID")
plt.show()


In [ ]:
print("""
Notes:
- Coordinates are synthetic; real analysis would map PULocationID/DOLocationID to actual lat/long centroids.
- Scaling factor (median) gives a crude conversion to miles; could use regression with intercept.
- Improve by: importing official NYC TLC taxi zone shapefile, computing centroids, projecting to a planar CRS, then L1.
""")